In [1]:
print("Hello World")

Hello World


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    f1_score, roc_auc_score
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [3]:
df = pd.read_csv('loan-cleaned.csv')
print(df.shape)
df.head()

(614, 12)


,Married,Dependents,Education,Self_Employed,LoanAmount,Loan_Amount_Term,Credit_History,Loan_Status,Total_Income,Gender_Male,Property_Area_Semiurban,Property_Area_Urban
0,0,0,1,0,128.0,360.0,1.0,1,5849.0,1,0,1
1,1,1,1,0,128.0,360.0,1.0,0,6091.0,1,0,0
2,1,0,1,1,66.0,360.0,1.0,1,3000.0,1,0,1
3,1,0,0,0,120.0,360.0,1.0,1,4941.0,1,0,1
4,0,0,1,0,141.0,360.0,1.0,1,6000.0,1,0,1


In [4]:
print(df['Loan_Status'].value_counts())

Loan_Status
1    422
0    192
Name: count, dtype: int64


In [ ]:
X = df.drop('Loan_Status', axis=1)
y = df['Loan_Status']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [7]:
# Handle imbalance for XGBoost
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
spw = neg / pos

print(f'scale_pos_weight: {spw:.2f}')

models = {
    "LogReg": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            class_weight="balanced", C=0.5, max_iter=3000, random_state=42
        ))
    ]),

    "DT": DecisionTreeClassifier(
        max_depth=5, min_samples_split=10,
        class_weight="balanced", random_state=42
    ),

    "RF": RandomForestClassifier(
        n_estimators=300, max_depth=6, min_samples_split=5,
        class_weight="balanced", random_state=42
    ),

    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(
            kernel="rbf", class_weight="balanced",
            probability=True, random_state=42
        ))
    ]),

    "XGB": XGBClassifier(
        scale_pos_weight=spw, n_estimators=200, max_depth=4,
        learning_rate=0.05, eval_metric="logloss", random_state=42
    ),

    "LGBM": LGBMClassifier(
        class_weight="balanced", n_estimators=200,
        learning_rate=0.05, max_depth=4, random_state=42
    )
}


scale_pos_weight: 0.46


In [9]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

for name, model in models.items():
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    probs = (model.predict_proba(X_test)[:, 1]
             if hasattr(model, "predict_proba")
             else model.decision_function(X_test))

    test_acc = accuracy_score(y_test, preds)
    test_f1  = f1_score(y_test, preds, average="weighted")
    test_auc = roc_auc_score(y_test, probs)

    cv_scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy")
    cv_mean, cv_std = cv_scores.mean(), cv_scores.std()

    results.append({
        "Model": name,
        "Acc": round(test_acc, 4),
        "F1": round(test_f1, 4),
        "AUC": round(test_auc, 4),
        "CV": round(cv_mean, 4),
        "Std": round(cv_std, 4),
        "Gap": round(test_acc - cv_mean, 4)
    })

results_df = pd.DataFrame(results).sort_values("F1", ascending=False)
results_df

[LightGBM] [Info] Number of positive: 337, number of negative: 154
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000312 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 289
[LightGBM] [Info] Number of data points in the train set: 491, number of used features: 11
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[

,Model,Acc,F1,AUC,CV,Std,Gap
2,RF,0.8374,0.8296,0.8406,0.7965,0.0158,0.0409
0,LogReg,0.8293,0.8272,0.8598,0.7509,0.0320,0.0784
3,SVM,0.8130,0.8072,0.8167,0.7427,0.0189,0.0703
4,XGB,0.7967,0.7999,0.8279,0.7508,0.0136,0.0459
5,LGBM,0.7805,0.7839,0.8192,0.7345,0.0234,0.0460
1,DT,0.7480,0.7427,0.7110,0.7492,0.0510,-0.0013


In [18]:
# Balance Score
results_df["Balance"] = (
    results_df["F1"] *
    results_df["AUC"] *
    (1 - results_df["Gap"].abs())
)

best = results_df.sort_values("Balance", ascending=False).iloc[0]

print("Best Model:", best["Model"])
print("Score:", round(best["Balance"], 4))
print("F1:", round(best["F1"], 4))
print("AUC:", round(best["AUC"], 4))
print("CV:", round(best["CV"], 4))
print("Gap:", round(best["Gap"], 4))



Best Model: RF
Score: 0.6688
F1: 0.8296
AUC: 0.8406
CV: 0.7965
Gap: 0.0409


In [ ]:
best_name = results_df.sort_values("Balance", ascending=False).iloc[0]["Model"]

print("Best model selected:", best_name)

Best model selected: RF


In [22]:
best_model = models[best_name]
best_model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",6
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",5
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y_

In [23]:
# Step 2: Get prediction probabilities
if hasattr(best_model, "predict_proba"):
    probs = best_model.predict_proba(X_test)[:, 1]
else:
    raw = best_model.decision_function(X_test)
    probs = (raw - raw.min()) / (raw.max() - raw.min())  # scale to 0–1

In [ ]:
# # Step 3: Default prediction (threshold = 0.5)
# default_preds = (probs >= 0.5).astype(int)
# default_f1 = f1_score(y_test, default_preds, average="weighted")

# print("Default Threshold F1:", round(default_f1, 4))

Default Threshold F1: 0.8296


In [ ]:
# # Step 4: Try different thresholds
# thresholds = np.arange(0.3, 0.8, 0.01)

# best_thresh = 0.5
# best_f1 = 0
# # Step 6: Print result
# print("Best Threshold:", round(best_thresh, 2))
# print("Best F1 Score:", round(best_f1, 4))

Best Threshold: 0.5
Best F1 Score: 0


In [28]:
thresholds = np.arange(0.3, 0.8, 0.01)
best_thresh, best_f1 = 0.5, 0.0

for t in thresholds:
    preds_t = (probs >= t).astype(int)
    f = f1_score(y_test, preds_t, average='weighted')
    if f > best_f1:
        best_f1, best_thresh = f, t

print(f'Optimal threshold : {best_thresh:.2f}')
print(f'Optimal threshold F1 : {best_f1:.4f}')

Optimal threshold : 0.43
Optimal threshold F1 : 0.8504


In [29]:
from sklearn.metrics import confusion_matrix

# Default threshold
cm_default = confusion_matrix(y_test, (probs >= 0.5).astype(int))

# Optimal threshold
cm_optimal = confusion_matrix(y_test, (probs >= best_thresh).astype(int))

print("Model:", best_name)

print("\nDefault Threshold (0.50):")
print(cm_default)

print("\nOptimal Threshold:", round(best_thresh, 2))
print(cm_optimal)

Model: RF

Default Threshold (0.50):
[[23 15]
 [ 5 80]]

Optimal Threshold: 0.43
[[22 16]
 [ 1 84]]


In [ ]:

preds_optimal = (probs >= best_thresh).astype(int)

print("Final Model:", best_name)
print("Threshold:", round(best_thresh, 2))

print("\nClassification Report:")
print(classification_report(
    y_test,
    preds_optimal,
    target_names=["Not Approved", "Approved"]
))

Final Model: RF
Threshold: 0.43

Classification Report:
              precision    recall  f1-score   support

Not Approved       0.96      0.58      0.72        38
    Approved       0.84      0.99      0.91        85

    accuracy                           0.86       123
   macro avg       0.90      0.78      0.81       123
weighted avg       0.88      0.86      0.85       123



In [38]:
comparison = results_df[[
    "Model", "F1", "AUC", "CV", "Gap", "Balance"
]].sort_values("Balance", ascending=False)


print(comparison)

    Model      F1     AUC      CV     Gap   Balance
2      RF  0.8296  0.8406  0.7965  0.0409  0.668840
0  LogReg  0.8272  0.8598  0.7509  0.0784  0.655466
4     XGB  0.7999  0.8279  0.7508  0.0459  0.631841
3     SVM  0.8072  0.8167  0.7427  0.0703  0.612896
5    LGBM  0.7839  0.8192  0.7345  0.0460  0.612631
1      DT  0.7427  0.7110  0.7492 -0.0013  0.527373
